

                          © 2026 NOOB DEV                                   
                       ALL RIGHTS RESERVED                                  
                                                                              
  This material is the **exclusive intellectual property of Noob Dev**.          
  **Strictly prohibited** to distribute, share, publish, reproduce, or           
  transmit this content — in whole or in part — outside of Noob Dev          
  sessions, platforms, or enrolled students.                                 
                                                                            
  Unauthorized sharing of this assignment outside the Noob Dev               
  community is a **violation of program terms** and may result in                
  immediate removal + legal actions from the Company.                                        

# CAIEP L1 : AI Personal Finance Tracker
## Noob Dev — Certified AI Engineer Professional L1

---

### ⏰ Deadline: **April 30, 2026 — 1 PM (SL TIME)**

---

This is your first real AI **agent** project. You are going to build a fully functional, chat-based personal finance tracker powered by a local AI agent with tool calling, a persistent SQLite database, and a Gradio user interface.

**This is not a tutorial.** You are expected to apply everything you have learned over the past two weeks and figure out implementation details on your own. That is how real engineering works.

> No OpenAI API key required. This project must run 100% free using **Groq**, **Ollama** or **OpenRouter** (If you want, you can go ahead with a paid OpenAI model, but it’s totally optional.).

---
## 🎯 Project Overview

You will build a **chat-first personal finance assistant** that users interact with entirely through natural language — no forms, no dropdowns, no spreadsheets. Just conversation.

The AI agent will:
- **Understand** what the user is saying (spending money, checking budget, etc.)
- **Decide** which tool to call based on the message
- **Execute** the tool (read or write to the database)
- **Respond** with a helpful, friendly message

### Core User Flows

| User says... | AI does... |
|---|---|
| `"My salary this month is $3,000"` | Calls `set_salary()` → saves to DB |
| `"Just paid $800 for rent"` | Calls `log_expense()` → saves to DB |
| `"Grabbed lunch for $12.50"` | Calls `log_expense()` → saves to DB |
| `"How much do I have left?"` | Calls `get_balance()` → reads DB → responds |
| `"Show me my spending breakdown"` | Calls `get_expense_summary()` → reads DB → responds |

### Example Conversation

```
User:  My monthly salary is $3,000
AI:    Got it! I've saved your April 2026 salary as $3,000.00.

User:  Paid rent today — $900
AI:    Logged $900.00 for Rent. Spent so far: $900.00 | Remaining: $2,100.00

User:  Bought groceries, spent $67.50 at the supermarket
AI:    Logged $67.50 for Groceries. Spent so far: $967.50 | Remaining: $2,032.50

User:  How much is left for the month?
AI:    Here's your April snapshot:
       Salary: $3,000.00 | Spent: $967.50 | Remaining: $2,032.50

User:  Break down my spending by category
AI:    April Spending Breakdown:
       - Rent:       $900.00  (30.0% of salary)
       - Groceries:  $67.50   (2.3% of salary)
       Total spent: $967.50 | Remaining: $2,032.50
```

---
## 🛠️ Tech Stack

| Component | Tool | Notes |
|-----------|------|-------|
| LLM | Groq (free tier), OpenRouter or Ollama (local) | Must support tool calling |
| UI | Gradio | Learned in Week 2  |
| Database | SQLite3 (built-in Python) | Learned in Week 2  |
| Tool Calling | OpenAI-compatible format | Learned in Week 2  |
| HTTP Client | OpenAI Python library | Same library, different `base_url` |

### Recommended Free Models

**Option A — Groq (recommended):**
- Sign up free at [console.groq.com](https://console.groq.com) — no credit card needed
- Model: `llama-3.3-70b-versatile`
- Fast, reliable, excellent tool calling support

**Option B — Ollama (fully local, no internet needed):**
- Install from [ollama.com](https://ollama.com)
- Run: `ollama pull llama3.1` in your terminal
- Model: `llama3.1` (use `llama3.1`, not `llama3.2` — tool calling support is better)

> ⚠️ **Important:** Not all models support tool calling. Use the recommended models above. If your tools are never being called, you are likely using the wrong model.

> ⚠️ **Important:** Add all the dependencies to the requirements.txt file.

---
## ⚙️ Step 1: Imports & Model Configuration

Set up your imports and configure which model/provider you are using.

---
## 🗄️ Step 1: Database Setup

Your SQLite database needs **two tables** to store salary and expense data.

### Table 1: `salary`
Stores the user's monthly income.

| Column | Type | Description |
|--------|------|-------------|
| `id` | INTEGER PRIMARY KEY | Auto-increments |
| `amount` | REAL | Monthly salary amount |
| `month` | TEXT | Month name — e.g. `"April"` |
| `year` | INTEGER | Year — e.g. `2026` |
| `created_at` | TEXT | Timestamp when recorded |

### Table 2: `expenses`
Stores every expense the user logs.

| Column | Type | Description |
|--------|------|-------------|
| `id` | INTEGER PRIMARY KEY | Auto-increments |
| `amount` | REAL | Expense amount |
| `category` | TEXT | e.g. `"Groceries"`, `"Transport"`, `"Rent"` |
| `description` | TEXT | Short description of the expense |
| `date` | TEXT | Date of the expense (`YYYY-MM-DD`) |
| `created_at` | TEXT | Timestamp when logged |


> 💡 **Hint:** You can use Gen AI tools for setup database and tables its totally fine only for this section.

---
## 🔧 Step 2: Implement the Tool Functions

You need **4 Python functions** that the AI agent can call. Each function must:
1. Connect to SQLite and do its job (read or write data)
2. Close the connection when done
3. Return a **string** result — the AI reads this string and uses it to compose its reply

---

### Tool 1 — `set_salary(amount, month, year)`
**When called:** User mentions their income, salary, or earnings for a month.
**What it does:** Inserts or replaces the salary record for that month in the database.
**Returns:** A confirmation string, e.g. `"Salary of $3,000.00 for April 2026 saved."`

---

### Tool 2 — `log_expense(amount, category, description, date)`
**When called:** User mentions spending money on anything.
**What it does:** Inserts a new expense row into the database. If `date` is not given, use today.
**Returns:** A confirmation string that also includes the updated balance — e.g. `"Logged $67.50 for Groceries. Spent: $967.50 | Remaining: $2,032.50"`

---

### Tool 3 — `get_balance()`
**When called:** User asks how much money is left, what their balance is, or how they are tracking.
**What it does:** Queries the salary and total expenses for the **current month**, calculates remaining.
**Returns:** A formatted string with salary, total spent, and remaining balance.

---

### Tool 4 — `get_expense_summary()`
**When called:** User asks for a breakdown, summary, or wants to know where their money went.
**What it does:** Groups all expenses for the current month by category, calculates totals and percentages.
**Returns:** A formatted string showing each category, amount, and percentage of salary.

> ⚠️ **Important rules for all functions:**
> - Always call `conn.close()` after you are done — never leave a connection open
> - Always call `conn.commit()` after any INSERT or UPDATE
> - Never put user values directly into SQL strings — always use `?` placeholders to prevent SQL injection

---
## 📋 Step 3: Describe Your Tools to the AI

The AI model does not know your Python functions exist. You have to describe them in a specific JSON format so the model knows:
- What tools are available
- When to use each one
- What parameters each tool expects

This is the **exact same format** used in **Week 2** for the airline support bot. Go back and look at that notebook — the `tools` list structure is identical.

Each tool description has three parts:
1. **`name`** — must exactly match your Python function name
2. **`description`** — tells the AI *when* to call this tool (write this carefully!)
3. **`parameters`** — describes each argument the function needs

> 💡 The `description` field is crucial. If you write a vague description, the AI will not know when to use the tool. Be specific about what triggers each tool call.

---
## 🔄 Step 4: The Tool Calling Loop

This is the **core logic** of your AI agent. When the user sends a message, your code must:

```
  [User message]
       │
       ▼
  Send to AI with tools list
       │
       ▼
  Did AI request a tool call?
  ├── YES → Extract tool name & args
  │          Call Python function
  │          Append result to messages
  │          Send back to AI  ──────────────┐
  │                                         │ (loop repeats)
  └── NO  → Return AI text response         │
             to user                        │
```

---
## 🖥️ Step 5: Build the Gradio UI

Your UI must use Gradio, and you can freely customize it as you like since it’s your project.

---
## 🧪 Test Conversations

Once your app is running, test it with all of the following inputs. Your app must handle every single one correctly for full marks.

### 1. Setting Up Income
```
My monthly salary is $3,500
I earn $4,000 a month
Set my income to 2800
```

### 2. Logging Expenses (natural language variety)
```
Spent $12 on coffee this morning
Paid rent — $900
Bought groceries for $67.50 at Walmart
Grabbed lunch with a friend, cost me $23
Netflix subscription renewed, $15.99
Filled up my gas tank, it was $55
Doctor appointment copay $30
```

### 3. Checking Balance
```
How much do I have left?
What's my remaining budget?
Am I on track this month?
How much have I spent so far?
```

### 4. Getting Summary
```
Show me my spending breakdown
Where am I spending the most?
Give me a summary of this month's expenses
What categories have I spent on?
```

> ✅ If all of these work, your agent is properly wired up.

---
## 📊 Evaluation Rubric

| Section | Marks | What We Are Looking For |
|---------|-------|-------------------------|
| Database setup | **10** | Tables created with correct columns and types |
| Tool implementations | **25** | All functions work correctly — read and write to DB accurately |
| Tool JSON definitions | **10** | All tools described with clear descriptions and correct parameter schemas |
| Tool calling loop | **20** | AI correctly decides which tool to call; loop handles multi-step calls and terminates properly |
| Gradio UI layout | **20** | Proper UI |
| Code quality | **10** | Clean, readable variable names; no dead code; no hardcoded secrets |
| README + submission | **5** | Clear setup instructions and working screenshot |
| **Total** | **100** | |

---
## 📤 Submission Instructions

**🗓 Deadline: April 30, 2026 — 1:00 PM (Sri Lanka Time)**  
Submissions after this time will **NOT be accepted under any circumstances**.

---

## 📦 Submission Components

You are required to submit **ALL of the following components**:

### 1️⃣ Full Project (ZIP File)

Submit a **single `.zip` file** with the exact structure below:

```
YourName_WhatsAppNumber_FinanceTracker/
├── your main application files (.ipynb)
├── your test database with some sample data in it
├── requirements.txt        ← all pip packages your project needs
└── README.md               ← setup and run instructions + screenshot

And all the necessary files, basically. 😉
```

---

### 2️⃣ 🎥 Video Demo (Mandatory)

You must create a **video demonstration** covering:

- Explanation of your **code structure**
- Key features and **logic of your implementation**
- **Live demo** of your application working end-to-end
- Keep it Under 10 min

> ⚠️ Make sure your video is clear, properly explained, and not rushed.

---

### 3️⃣ ☁️ Google Drive Submission

- Upload **both your ZIP file and Video Demo** to Google Drive  
- Set sharing permissions to: **“Anyone with the link can view”**  
- Double-check that your links are accessible without login issues  

---

### 4️⃣ 📝 Final Submission (Google Form)

- Submit your **Google Drive links** via the **Google Form provided**  
- Ensure:
  - Links are correct  
  - Files are accessible  
  - Submission is completed before the deadline  

---

## 🚫 Do NOT Include

- ❌ `.env` files or any **API keys / secrets**  
- ❌ `__pycache__` folders or unnecessary files  
- ❌ Large model files (e.g., Ollama models — keep them local)  
- ❌ Any irrelevant content  

---

## ⚠️ Submission Policy

> Late submissions will be **automatically rejected** after the deadline.  
> No extensions will be provided — plan and submit early.

```
╔══════════════════════════════════════════════════════════════════════════════╗
║  Good luck. Build something you would actually use.                         ║
║                                              — Noob Dev Team                ║
╚══════════════════════════════════════════════════════════════════════════════╝
```